# Исследование уровня лояльности клиентов телекоммуникационной компании 

# Описание 

Заказчик этого исследования — большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ. Перед компанией стоит задача определить текущий уровень потребительской лояльности, или NPS (от англ. Net Promoter Score), среди клиентов из России. 
Чтобы определить уровень лояльности, клиентам задавали классический вопрос: «Оцените по шкале от 1 до 10 вероятность того, что вы порекомендуете компанию друзьям и знакомым».

Компания провела опрос и попросила подготовить дашборд с его итогами. Большую базу данных для такой задачи разворачивать не стали и выгрузили данные в SQLite. 
Чтобы оценить результаты опроса, оценки обычно делят на три группы:

9-10 баллов — «cторонники» (англ. promoters), 7-8 баллов — «нейтралы» (англ. passives), 0-6 баллов — «критики» (англ. detractors).

Итоговое значение NPS рассчитывается по формуле: % «сторонников» - % «критиков».
Таким образом, значение этого показателя варьируется от -100% (когда все клиенты «критики») до 100% (когда все клиенты лояльны к сервису). Но это крайние случаи, которые редко встретишь на практике. 
Интерпретируя результаты NPS-опросов, следует также помнить, что само значение мало о чём говорит. Однако исследования показывают, что клиенты-сторонники полезны любому бизнесу. Они чаще других повторно совершают покупки, активнее тестируют обновления и приводят в сервис своих друзей и знакомых. Поэтому NPS остаётся одной из важнейших метрик бизнеса. 

# Цель исследования

Найти ответы на вопросы:
1. Как распределены участники опроса по возрасту и полу?
2. Каких пользователей больше: новых или старых?
3. Пользователи из каких городов активнее участвовали в опросе?
4. Какие группы пользователей наиболее лояльны к сервису? Какие менее?
5. Какой общий NPS среди всех опрошенных?
6. Как можно описать клиентов, которые относятся к группе cторонников (англ. promoters)?

Ответы на вопросы необходимо оформить в виде презентации на сайте Tableau Public

# Документация 

### Таблица user содержит основную информацию о клиентах.
1. user_id - Идентификатор клиента, первичный ключ таблицы
2. lt_day - Количество дней «жизни» клиента
3. age - возраст клиента в годах
4. gender_segment - пол клиента (1 – женщина, 0 – мужчина)
5. os_name	- тип операционной системы
6. cpe_type_name - тип устройства
7. location_id - идентификатор домашнего региона клиента, внешний ключ, отсылающий к таблице location
8. age_gr_id - идентификатор возрастного сегмента клиента, внешний ключ, отсылающий к таблице age_segment
9. tr_gr_id - идентификатор сегмента клиента по объёму потребляемого трафика в месяц, внешний ключ, отсылающий к таблице traffic_segment
10. lt_gr_id -  идентификатор сегмента клиента по количеству месяцев «жизни», внешний ключ, отсылающий к таблице lifetime_segment
11. nps_score - оценка клиента в NPS-опросе (от 1 до 10)

### Таблица location - справочник территорий, в которых телеком-компания оказывает услуги.
1. location_id - идентификатор записи, первичный ключ
2. country - страна
3. city - город

### Таблица age_segment - данные о возрастных сегментах клиентов.
1. age_gr_id - идентификатор сегмента, первичный ключ
2. bucket_min - минимальная граница сегмента
3. bucket_max - максимальная граница сегмента
4. title - название сегмента

### Таблица traffic_segment - данные о выделяемых сегментах по объёму потребляемого трафика.
1. tr_gr_id - идентификатор сегмента, первичный ключ
2. bucket_min - минимальная граница сегмента
3. bucket_max - максимальная граница сегмента
4. title - название сегмента

### Таблица lifetime_segment - данные о выделяемых сегментах по количеству месяцев «жизни» клиента — лайфтайму.
1. lt_gr_id - идентификатор сегмента, первичный ключ
2. bucket_min - минимальная граница сегмента
3. bucket_max - Максимальная граница сегмента
4. title - название сегмента

In [2]:
path_to_db_platform = "/datasets/telecomm_csi.db"
path_to_db = None

if os.path.exists(path_to_db_platform):
    path_to_db = path_to_db_platform # если путь на платформе ведёт к БД, то он становится итоговым
else:
    raise Exception("Файл с базой данных SQLite не найден!") # иначе выводится сообщение о том, что файл не найден

if path_to_db:
    engine = create_engine(f"sqlite:///{path_to_db}", echo=False) # создаём подключение к базе

In [3]:
query = text("""
    WITH data_cte AS (
    SELECT
        u.user_id,
        u.lt_day,
        u.age,
        u.gender_segment,
        u.os_name,
        u.cpe_type_name,
        l.country,
        l.city,
        SUBSTR(a.title, 3) AS age_segment,
        SUBSTR(t.title, 3) AS traffic_segment,
        SUBSTR(lt.title, 3) AS lifetime_segment,
        u.nps_score,
        CASE WHEN u.nps_score >= 9 THEN 'сторонники'
             WHEN u.nps_score >= 7 THEN 'нейтралы'
             ELSE 'критики'
        END AS nps_group,
        CASE WHEN u.lt_day <= 365 THEN 'True' ELSE 'False' END AS is_new
    FROM
        user AS u
        LEFT JOIN location AS l ON l.location_id = u.location_id
        LEFT JOIN age_segment AS a ON a.age_gr_id = u.age_gr_id
        LEFT JOIN traffic_segment AS t ON t.tr_gr_id = u.tr_gr_id
        LEFT JOIN lifetime_segment AS lt ON lt.lt_gr_id = u.lt_gr_id
)
SELECT *
FROM data_cte;
""") # создаём общую витрину из отдельных полей для дальнейшего перевода в csv

<div class="alert alert-block alert-success">✔️
    

__Комментарий от ревьюера №1__

Молодец, что используешь `SUBSTR`

In [4]:
df = pd.read_sql(query, engine)  # применение запроса 
# display(df)
df.to_csv("telecomm_csi.db", index = False)

Данные готовы к обработке

С результатами исследования можно ознакомиться по ссылке https://public.tableau.com/app/profile/evgeniy.goldshtein/viz/_17422303700950/sheet24